Gold_dailey_orders

In [0]:
%sql
SELECT CAST(order_timestamp AS DATE) AS order_date, COUNT(order_id) AS total_number_orders, SUM(item_count) AS total_items_per_date
FROM dp_750.default.silver_orders
GROUP BY order_date

In [0]:
%sql

CREATE OR REPLACE TABLE dp_750.default.gold_daily_orders AS (
SELECT CAST(order_timestamp AS DATE) AS order_date, COUNT(order_id) AS total_number_orders, SUM(item_count) AS total_items_per_date
FROM dp_750.default.silver_orders
GROUP BY order_date
) 





Gold_hourly_orders

In [0]:
%sql
SELECT CAST(order_timestamp AS DATE), HOUR(order_timestamp)AS hour_in_day, COUNT(order_id) AS total_number_orders, sum(item_count) total_items_per_date
FROM dp_750.default.silver_orders
GROUP BY CAST(order_timestamp AS DATE), HOUR(order_timestamp)


In [0]:
%sql
CREATE OR REPLACE TABLE dp_750.default.gold_hourly_orders AS (SELECT CAST(order_timestamp AS DATE), HOUR(order_timestamp)AS hour_in_day, COUNT(order_id) AS total_number_orders, sum(item_count) total_items_per_date
FROM dp_750.default.silver_orders
GROUP BY CAST(order_timestamp AS DATE), HOUR(order_timestamp))

Gold_orders_by_Zone

In [0]:
%sql
SELECT delivery_zone, CAST(order_timestamp AS DATE), COUNT(order_id) AS total_number_orders, SUM(item_count) AS total_items_per_day
FROM dp_750.default.silver_orders
GROUP BY delivery_zone, CAST(order_timestamp AS DATE)

In [0]:
%sql
CREATE OR REPLACE TABLE dp_750.default.gold_orders_by_zone AS
SELECT delivery_zone, CAST(order_timestamp AS DATE), COUNT(order_id) AS total_number_orders, SUM(item_count) AS total_items_per_day
FROM dp_750.default.silver_orders
GROUP BY delivery_zone, CAST(order_timestamp AS DATE)

Gold_daily_weather

In [0]:
%sql
SELECT
    observation_date,
    Bureau_of_Meteorology_station_number AS station_id,
    'maximum_temperature' AS measure,
    Maximum_temperature_Degree_C AS value,
    'degC' AS unit
FROM dp_750.default.silver_temperature

UNION ALL

SELECT
    observation_date,
    Bureau_of_Meteorology_station_number AS station_id,
    'rainfall' AS measure,
    Rainfall_amount_millimetres AS value,
    'mm' AS unit
FROM dp_750.default.silver_rainfall;

In [0]:
%sql
CREATE OR REPLACE TABLE dp_750.default.gold_daily_weather AS
SELECT
    observation_date,
    Bureau_of_Meteorology_station_number AS station_id,
    'maximum_temperature' AS measure,
    Maximum_temperature_Degree_C AS value,
    'degC' AS unit
FROM dp_750.default.silver_temperature

UNION ALL

SELECT
    observation_date,
    Bureau_of_Meteorology_station_number AS station_id,
    'rainfall' AS measure,
    Rainfall_amount_millimetres AS value,
    'mm' AS unit
FROM dp_750.default.silver_rainfall;

Gold_date_dimension

In [0]:
%sql
CREATE OR REPLACE TABLE dp_750.default.gold_dim_date AS

WITH order_dates AS (
    SELECT
        MIN(CAST(order_timestamp AS DATE)) AS start_date,
        MAX(CAST(order_timestamp AS DATE)) AS end_date
    FROM dp_750.default.silver_orders
),

calendar AS (
    SELECT EXPLODE(
        SEQUENCE(start_date, end_date, INTERVAL 1 DAY)
    ) AS date
    FROM order_dates
)

SELECT
    date,
    CAST(DATE_FORMAT(date, 'yyyyMMdd') AS INT) AS date_key,
    YEAR(date) AS year,
    MONTH(date) AS month_number,
    DATE_FORMAT(date, 'MMMM') AS month_name,
    DAYOFMONTH(date) AS day_of_month,
    PMOD(DAYOFWEEK(date) + 5, 7) + 1 AS day_of_week_number,
    DATE_FORMAT(date, 'EEEE') AS day_name,
    DAYOFWEEK(date) IN (1, 7) AS is_weekend
FROM calendar;

Gold_Zone_dimension

In [0]:
%sql
SELECT *
FROM dp_750.default.gold_dim_date
ORDER BY date;

In [0]:
%sql
CREATE OR REPLACE TABLE dp_750.default.gold_dim_zone AS

WITH location_info AS (
    SELECT * FROM VALUES
    (
        'Melbourne CBD',
        'The Hoddle Grid forms the historic heart of Melbourne CBD.',
        'Central Melbourne context; exact delivery boundaries are undefined.',
        'https://participate.melbourne.vic.gov.au/amendment-c386'
    ),
    (
        'Inner North',
        'Yarra includes inner-city suburbs such as Carlton North, Fitzroy North and Princes Hill.',
        'These suburbs provide illustrative context, not a complete zone boundary.',
        'https://www.yarracity.vic.gov.au/about-us/about-yarra'
    ),
    (
        'Northern Suburbs',
        'The Northern Metro planning region includes Hume, Darebin and Whittlesea.',
        'Northern Metro provides broad context; delivery boundaries are undefined.',
        'https://www.planning.vic.gov.au/__data/assets/pdf_file/0030/637860/northern_introduction_chapter03.pdf'
    ),
    (
        'Eastern Suburbs',
        'The Eastern Metro planning region includes Knox, Monash, Whitehorse and Yarra Ranges.',
        'Eastern Metro provides broad context; delivery boundaries are undefined.',
        'https://www.planning.vic.gov.au/__data/assets/pdf_file/0033/637872/eastern_introduction_chapter03.pdf'
    ),
    (
        'Western Suburbs',
        'The Western Metro planning region includes Brimbank, Maribyrnong, Melton and Wyndham.',
        'Western Metro provides broad context; delivery boundaries are undefined.',
        'https://www.planning.vic.gov.au/__data/assets/pdf_file/0031/637843/western_introduction_chapter03.pdf'
    ),
    (
        'South East Melbourne',
        'The Southern Metro planning region includes Casey, Cardinia and Greater Dandenong.',
        'Southern Metro is illustrative context; it does not define this synthetic delivery zone.',
        'https://www.planning.vic.gov.au/__data/assets/pdf_file/0032/637916/southern_introduction_chapter03.pdf'
    )
    AS locations(
        delivery_zone,
        location_description,
        mapping_assumption,
        source_url
    )
),

zones AS (
    SELECT DISTINCT delivery_zone
    FROM dp_750.default.silver_orders
)

SELECT
    z.delivery_zone,
    l.location_description,
    COALESCE(
        l.mapping_assumption,
        'Zone requires review; no location mapping established.'
    ) AS mapping_assumption,
    l.source_url,
    'No zone-specific weather station mapping established.'
        AS weather_mapping_note
FROM zones z
LEFT JOIN location_info l
    ON z.delivery_zone = l.delivery_zone;

In [0]:
%sql
SELECT * 
FROM dp_750.default.gold_dim_zone 